In [1]:
!unzip Final_Modeling_Dataset.zip

Streaming output truncated to the last 5000 lines.
  inflating: Final_Modeling_Dataset/train/blur/366_very_high.jpg  
  inflating: Final_Modeling_Dataset/train/blur/367_high.jpg  
  inflating: Final_Modeling_Dataset/train/blur/367_low.jpg  
  inflating: Final_Modeling_Dataset/train/blur/367_medium.jpg  
  inflating: Final_Modeling_Dataset/train/blur/367_very_high.jpg  
  inflating: Final_Modeling_Dataset/train/blur/368_high.jpg  
  inflating: Final_Modeling_Dataset/train/blur/368_low.jpg  
  inflating: Final_Modeling_Dataset/train/blur/368_medium.jpg  
  inflating: Final_Modeling_Dataset/train/blur/368_very_high.jpg  
  inflating: Final_Modeling_Dataset/train/blur/369_high.jpg  
  inflating: Final_Modeling_Dataset/train/blur/369_low.jpg  
  inflating: Final_Modeling_Dataset/train/blur/369_medium.jpg  
  inflating: Final_Modeling_Dataset/train/blur/369_very_high.jpg  
  inflating: Final_Modeling_Dataset/train/blur/36_high.jpg  
  inflating: Final_Modeling_Dataset/train/blur/36_low.jpg  

In [2]:
from IPython.display import display, Javascript
import time

def keep_awake():
    display(Javascript('''
        function ClickConnect(){
            console.log("Working");
            document.querySelector("colab-toolbar-button#connect").click()
        }
        setInterval(ClickConnect,60000)
    '''))

keep_awake()

<IPython.core.display.Javascript object>

In [3]:
!pip install lpips

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.8/53.8 kB 3.6 MB/s eta 0:00:00


In [4]:
!pip install facenet-pytorch --no-deps


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 23.0 MB/s eta 0:00:00


In [5]:
!pip install scikit-image


In [6]:
!pip install tqdm

In [7]:

import os
import time
from glob import glob

import numpy as np
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader

import torchvision.transforms as transforms

import lpips

from skimage.metrics import structural_similarity as ssim_metric

from facenet_pytorch import MTCNN



In [8]:

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print("Using device:", DEVICE)

BATCH_SIZE = 8
EPOCHS = 30
LEARNING_RATE = 1e-4
IMAGE_SIZE = 256


Using device: cuda


In [9]:
# dataset hasil unzip
DATASET_PATH = '/content/Final_Modeling_Dataset'

TRAIN_BLUR_DIR = f'{DATASET_PATH}/train/blur'
TRAIN_SHARP_DIR = f'{DATASET_PATH}/train/sharp'

VAL_BLUR_DIR = f'{DATASET_PATH}/validation/blur'
VAL_SHARP_DIR = f'{DATASET_PATH}/validation/sharp'

TEST_BLUR_DIR = f'{DATASET_PATH}/test/blur'
TEST_SHARP_DIR = f'{DATASET_PATH}/test/sharp'


In [10]:
# SAVE DIRECTORY

MODEL_SAVE_DIR = '/content/saved_models'
OUTPUT_DIR = '/content/output_images'

os.makedirs(MODEL_SAVE_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [11]:

transform = transforms.Compose([

    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.5, 0.5, 0.5],
        std=[0.5, 0.5, 0.5]
    )
])

In [12]:
class DeblurDataset(Dataset):

    def __init__(self, blur_dir, sharp_dir, transform=None):

        self.blur_dir = blur_dir
        self.sharp_dir = sharp_dir
        self.transform = transform

        self.blur_images = sorted(
            glob(os.path.join(blur_dir, '*'))
        )

    def __len__(self):

        return len(self.blur_images)

    # =====================================================
    # STEM-BASED MATCHING
    # =====================================================

    def get_sharp_name(self, blur_path):

        filename = os.path.basename(blur_path)

        stem = os.path.splitext(filename)[0]

        # REMOVE BLUR LEVEL
        if '_very_high' in stem:

            stem = stem.replace('_very_high', '')

        elif '_medium' in stem:

            stem = stem.replace('_medium', '')

        elif '_high' in stem:

            stem = stem.replace('_high', '')

        elif '_low' in stem:

            stem = stem.replace('_low', '')

        sharp_name = stem + '.jpg'

        return sharp_name

    def __getitem__(self, idx):

        blur_path = self.blur_images[idx]

        sharp_name = self.get_sharp_name(blur_path)

        sharp_path = os.path.join(
            self.sharp_dir,
            sharp_name
        )

        # CHECK FILE EXIST
        if not os.path.exists(sharp_path):

            raise FileNotFoundError(
                f'Sharp image not found: {sharp_path}'
            )

        blur_img = Image.open(
            blur_path
        ).convert('RGB')

        sharp_img = Image.open(
            sharp_path
        ).convert('RGB')

        if self.transform:

            blur_img = self.transform(blur_img)

            sharp_img = self.transform(sharp_img)

        return blur_img, sharp_img

In [13]:
class DoubleConv(nn.Module):

    def __init__(self, in_channels, out_channels):

        super().__init__()

        self.conv = nn.Sequential(

            nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size=3,
                padding=1
            ),

            nn.BatchNorm2d(out_channels),

            nn.ReLU(inplace=True),

            nn.Conv2d(
                out_channels,
                out_channels,
                kernel_size=3,
                padding=1
            ),

            nn.BatchNorm2d(out_channels),

            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.conv(x)

In [14]:
class UNet(nn.Module):

    def __init__(self):

        super().__init__()

        # ENCODER
        self.down1 = DoubleConv(3, 64)
        self.pool1 = nn.MaxPool2d(2)

        self.down2 = DoubleConv(64, 128)
        self.pool2 = nn.MaxPool2d(2)

        self.down3 = DoubleConv(128, 256)
        self.pool3 = nn.MaxPool2d(2)

        self.down4 = DoubleConv(256, 512)
        self.pool4 = nn.MaxPool2d(2)

        # BOTTLENECK
        self.bottleneck = DoubleConv(512, 1024)

        # DECODER
        self.up1 = nn.ConvTranspose2d(
            1024,
            512,
            kernel_size=2,
            stride=2
        )

        self.conv1 = DoubleConv(1024, 512)

        self.up2 = nn.ConvTranspose2d(
            512,
            256,
            kernel_size=2,
            stride=2
        )

        self.conv2 = DoubleConv(512, 256)

        self.up3 = nn.ConvTranspose2d(
            256,
            128,
            kernel_size=2,
            stride=2
        )

        self.conv3 = DoubleConv(256, 128)

        self.up4 = nn.ConvTranspose2d(
            128,
            64,
            kernel_size=2,
            stride=2
        )

        self.conv4 = DoubleConv(128, 64)

        self.final = nn.Conv2d(
            64,
            3,
            kernel_size=1
        )

    def forward(self, x):

        d1 = self.down1(x)
        p1 = self.pool1(d1)

        d2 = self.down2(p1)
        p2 = self.pool2(d2)

        d3 = self.down3(p2)
        p3 = self.pool3(d3)

        d4 = self.down4(p3)
        p4 = self.pool4(d4)

        bn = self.bottleneck(p4)

        up1 = self.up1(bn)
        up1 = torch.cat([up1, d4], dim=1)
        up1 = self.conv1(up1)

        up2 = self.up2(up1)
        up2 = torch.cat([up2, d3], dim=1)
        up2 = self.conv2(up2)

        up3 = self.up3(up2)
        up3 = torch.cat([up3, d2], dim=1)
        up3 = self.conv3(up3)

        up4 = self.up4(up3)
        up4 = torch.cat([up4, d1], dim=1)
        up4 = self.conv4(up4)

        output = self.final(up4)

        output = torch.tanh(output)

        return output

In [15]:
def tensor_to_numpy(img_tensor):

    img = img_tensor.detach().cpu()

    img = (img + 1) / 2

    img = torch.clamp(img, 0, 1)

    img = img.permute(1, 2, 0).numpy()

    return img


def calculate_psnr(pred, target):

    mse = torch.mean((pred - target) ** 2)

    if mse == 0:
        return 100

    psnr = 20 * torch.log10(
        2.0 / torch.sqrt(mse)
    )

    return psnr.item()


def calculate_ssim(pred, target):

    pred_np = tensor_to_numpy(pred)
    target_np = tensor_to_numpy(target)

    score = ssim_metric(
        target_np,
        pred_np,
        channel_axis=2,
        data_range=1.0
    )

    return score

In [16]:
train_dataset = DeblurDataset(
    TRAIN_BLUR_DIR,
    TRAIN_SHARP_DIR,
    transform
)

val_dataset = DeblurDataset(
    VAL_BLUR_DIR,
    VAL_SHARP_DIR,
    transform
)

test_dataset = DeblurDataset(
    TEST_BLUR_DIR,
    TEST_SHARP_DIR,
    transform
)


In [17]:

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2
)

val_loader = DataLoader(
    val_dataset,
    batch_size=1,
    shuffle=False,
    num_workers=2
)

test_loader = DataLoader(
    test_dataset,
    batch_size=1,
    shuffle=False,
    num_workers=2
)


In [18]:
print('='*50)

print(f'Train Images      : {len(train_dataset)}')

print(f'Validation Images : {len(val_dataset)}')

print(f'Test Images       : {len(test_dataset)}')

print('='*50)

Train Images      : 7048
Validation Images : 612
Test Images       : 624


In [19]:
import torch

print(torch.cuda.is_available())

print(torch.cuda.get_device_name(0))

True
Tesla T4


In [20]:
model = UNet().to(DEVICE)

criterion = nn.L1Loss()

optimizer = optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE
)

lpips_metric = lpips.LPIPS(
    net='alex'
).to(DEVICE)

# =====================================================
# PARAMETER COUNT
# =====================================================

num_params = sum(
    p.numel()
    for p in model.parameters()
)

print(f'Total Parameters: {num_params:,}')

# =====================================================
# TRAIN FUNCTION
# =====================================================

def train_epoch(loader):

    model.train()

    total_loss = 0

    progress_bar = tqdm(
        loader,
        desc='Training',
        leave=False
    )

    for blur, sharp in progress_bar:

        blur = blur.to(DEVICE)
        sharp = sharp.to(DEVICE)

        optimizer.zero_grad()

        output = model(blur)

        loss = criterion(output, sharp)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

# =====================================================
# VALIDATION FUNCTION
# =====================================================

def validate_epoch(loader):

    model.eval()

    total_loss = 0
    total_psnr = 0
    total_ssim = 0

    progress_bar = tqdm(
        loader,
        desc='Validation',
        leave=False
    )

    with torch.no_grad():

        for blur, sharp in progress_bar:

            blur = blur.to(DEVICE)
            sharp = sharp.to(DEVICE)

            output = model(blur)

            loss = criterion(output, sharp)

            total_loss += loss.item()

            psnr = calculate_psnr(
                output[0],
                sharp[0]
            )

            ssim = calculate_ssim(
                output[0],
                sharp[0]
            )

            total_psnr += psnr
            total_ssim += ssim

    return (
        total_loss / len(loader),
        total_psnr / len(loader),
        total_ssim / len(loader)
    )

# =====================================================
# TRAINING LOOP
# =====================================================

best_psnr = 0
best_model_path = ''

for epoch in range(EPOCHS):

    print('\n' + '='*50)

    print(f'Epoch [{epoch+1}/{EPOCHS}]')

    print('='*50)

    start_time = time.time()

    # =============================================
    # TRAIN
    # =============================================

    train_loss = train_epoch(train_loader)

    # =============================================
    # VALIDATION
    # =============================================

    val_loss, val_psnr, val_ssim = validate_epoch(val_loader)

    epoch_time = time.time() - start_time

    # =============================================
    # PRINT RESULT
    # =============================================

    print(f'Train Loss : {train_loss:.4f}')

    print(f'Val Loss   : {val_loss:.4f}')

    print(f'Val PSNR   : {val_psnr:.4f}')

    print(f'Val SSIM   : {val_ssim:.4f}')

    print(f'Time/Epoch : {epoch_time:.2f} sec')

    # =============================================
    # SAVE EVERY MODEL
    # =============================================

    model_save_path = os.path.join(
        MODEL_SAVE_DIR,
        f'unet_epoch_{epoch+1}.pth'
    )

    torch.save(
        model.state_dict(),
        model_save_path
    )

    print(f'Model saved: {model_save_path}')


    # SAVE BEST MODEL

    if val_psnr > best_psnr:

        best_psnr = val_psnr

        best_model_path = model_save_path

        print('Best model updated!')

Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=AlexNet_Weights.IMAGENET1K_V1`. You can also use `weights=AlexNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/alexnet-owt-7be5be79.pth" to /root/.cache/torch/hub/checkpoints/alexnet-owt-7be5be79.pth


100%|██████████| 233M/233M [00:01<00:00, 158MB/s]


Loading model from: /usr/local/lib/python3.12/dist-packages/lpips/weights/v0.1/alex.pth
Total Parameters: 31,043,651

Epoch [1/30]


Train Loss : 0.1242
Val Loss   : 0.0745
Val PSNR   : 25.0093
Val SSIM   : 0.7300
Time/Epoch : 448.84 sec
Model saved: /content/saved_models/unet_epoch_1.pth
Best model updated!

Epoch [2/30]


... (25 outputs left out)

Train Loss : 0.0423
Val Loss   : 0.0371
Val PSNR   : 30.7682
Val SSIM   : 0.8878
Time/Epoch : 452.51 sec
Model saved: /content/saved_models/unet_epoch_27.pth
Best model updated!

Epoch [28/30]


Train Loss : 0.0423
Val Loss   : 0.0385
Val PSNR   : 30.6092
Val SSIM   : 0.8822
Time/Epoch : 453.07 sec
Model saved: /content/saved_models/unet_epoch_28.pth

Epoch [29/30]


Train Loss : 0.0414
Val Loss   : 0.0394
Val PSNR   : 30.4900
Val SSIM   : 0.8882
Time/Epoch : 452.97 sec
Model saved: /content/saved_models/unet_epoch_29.pth

Epoch [30/30]


Train Loss : 0.0418
Val Loss   : 0.0402
Val PSNR   : 30.5793
Val SSIM   : 0.8318
Time/Epoch : 452.75 sec
Model saved: /content/saved_models/unet_epoch_30.pth


In [21]:
import os

blur_dir = '/content/Final_Modeling_Dataset/train/blur'
sharp_dir = '/content/Final_Modeling_Dataset/train/sharp'

print(os.listdir(blur_dir)[:10])

print(os.listdir(sharp_dir)[:10])

['1738_medium.jpg', '1232_very_high.jpg', '1384_low.jpg', '1965_very_high.jpg', '1956_medium.jpg', '1481_high.jpg', '1458_very_high.jpg', '435_very_high.jpg', '199_medium.jpg', '976_high.jpg']
['1021.jpg', '1832.jpg', '1475.jpg', '1610.jpg', '543.jpg', '167.jpg', '1428.jpg', '305.jpg', '798.jpg', '507.jpg']


In [22]:
print('\nLoading best model...')

print(f'Best Model: {best_model_path}')

model.load_state_dict(
    torch.load(best_model_path)
)

model.eval()

all_psnr = []
all_ssim = []
all_lpips = []

inference_times = []

with torch.no_grad():

    for idx, (blur, sharp) in enumerate(tqdm(test_loader)):

        blur = blur.to(DEVICE)
        sharp = sharp.to(DEVICE)

        # =============================================
        # INFERENCE
        # =============================================

        start = time.time()

        output = model(blur)

        end = time.time()

        inference_times.append(end - start)

        # =============================================
        # SAVE OUTPUT IMAGE
        # =============================================

        output_img = tensor_to_numpy(output[0])

        output_img = (output_img * 255).astype(np.uint8)

        output_pil = Image.fromarray(output_img)

        # pakai nama asli blur image
        filename = os.path.basename(
            test_dataset.blur_images[idx]
        )

        save_name = f'deblur_{filename}'

        output_pil.save(
            os.path.join(
                OUTPUT_DIR,
                save_name
            )
        )

        # =============================================
        # METRICS
        # =============================================

        psnr = calculate_psnr(
            output[0],
            sharp[0]
        )

        ssim_score = calculate_ssim(
            output[0],
            sharp[0]
        )

        lpips_score = lpips_metric(
            output,
            sharp
        )

        all_psnr.append(psnr)

        all_ssim.append(ssim_score)

        all_lpips.append(
            lpips_score.item()
        )

print('\n===== FINAL RESULTS =====')

print(f'PSNR  : {np.mean(all_psnr):.4f}')

print(f'SSIM  : {np.mean(all_ssim):.4f}')

print(f'LPIPS : {np.mean(all_lpips):.4f}')

print(
    f'Inference Time/Image : '
    f'{np.mean(inference_times):.4f} sec'
)

print('\nAll deblurred images saved!')


Loading best model...
Best Model: /content/saved_models/unet_epoch_27.pth


100%|██████████| 624/624 [00:28<00:00, 22.15it/s]


===== FINAL RESULTS =====
PSNR  : 29.8644
SSIM  : 0.8766
LPIPS : 0.1398
Inference Time/Image : 0.0064 sec

All deblurred images saved!


In [26]:
mtcnn = MTCNN(
    keep_all=True,
    device=DEVICE
)

# =====================================================
# FACE DETECTION PER BLUR LEVEL
# =====================================================

results = {

    'low': {
        'blur': 0,
        'deblur': 0
    },

    'medium': {
        'blur': 0,
        'deblur': 0
    },

    'high': {
        'blur': 0,
        'deblur': 0
    },

    'very_high': {
        'blur': 0,
        'deblur': 0
    }
}

with torch.no_grad():

    for idx, (blur, sharp) in enumerate(tqdm(test_loader)):

        blur = blur.to(DEVICE)

        output = model(blur)

        # =============================================
        # CONVERT IMAGE
        # =============================================

        blur_img = tensor_to_numpy(blur[0])
        output_img = tensor_to_numpy(output[0])

        blur_img = (blur_img * 255).astype(np.uint8)
        output_img = (output_img * 255).astype(np.uint8)

        # =============================================
        # GET BLUR LEVEL FROM FILENAME
        # =============================================

        blur_path = test_dataset.blur_images[idx]

        filename = os.path.basename(blur_path)

        if 'very_high' in filename:
            level = 'very_high'

        elif 'high' in filename:
            level = 'high'

        elif 'medium' in filename:
            level = 'medium'

        elif 'low' in filename:
            level = 'low'

        else:
            continue

        # =============================================
        # FACE DETECTION
        # =============================================

        blur_boxes, _ = mtcnn.detect(
            Image.fromarray(blur_img)
        )

        output_boxes, _ = mtcnn.detect(
            Image.fromarray(output_img)
        )

        # =============================================
        # COUNT FACES
        # =============================================

        blur_count = 0
        output_count = 0

        if blur_boxes is not None:
            blur_count = len(blur_boxes)

        if output_boxes is not None:
            output_count = len(output_boxes)

        # =============================================
        # SAVE RESULT
        # =============================================

        results[level]['blur'] += blur_count
        results[level]['deblur'] += output_count

        # =============================================
        # PRINT PER IMAGE
        # =============================================

        print(f'\nImage: {filename}')

        print(f'Blur Level : {level}')

        print(
            f'Blur Faces Detected   : '
            f'{blur_count}'
        )

        print(
            f'Deblur Faces Detected : '
            f'{output_count}'
        )

        print('-----------------------------------')

# =====================================================
# FINAL RESULT
# =====================================================

print('\n===== FACE DETECTION RESULTS =====')

for level in results:

    blur_faces = results[level]['blur']

    deblur_faces = results[level]['deblur']

    improvement = 0

    if blur_faces > 0:

        improvement = (
            (
                deblur_faces - blur_faces
            )
            / blur_faces
        ) * 100

    print(f'\nBLUR LEVEL : {level.upper()}')

    print(
        f'Blur Faces Detected   : '
        f'{blur_faces}'
    )

    print(
        f'Deblur Faces Detected : '
        f'{deblur_faces}'
    )

    print(
        f'Improvement           : '
        f'{improvement:.2f}%'
    )

    print('===================================')

  0%|          | 2/624 [00:00<03:50,  2.70it/s]


Image: 0_high.jpg
Blur Level : high
Blur Faces Detected   : 1
Deblur Faces Detected : 1
-----------------------------------

Image: 0_low.jpg
Blur Level : low
Blur Faces Detected   : 1
Deblur Faces Detected : 1
-----------------------------------


  1%|          | 4/624 [00:01<02:37,  3.94it/s]


Image: 0_medium.jpg
Blur Level : medium
Blur Faces Detected   : 1
Deblur Faces Detected : 1
-----------------------------------

Image: 0_very_high.jpg
Blur Level : very_high
Blur Faces Detected   : 0
Deblur Faces Detected : 1
-----------------------------------


  1%|          | 6/624 [00:01<02:18,  4.47it/s]


Image: 100_high.jpg
Blur Level : high
Blur Faces Detected   : 1
Deblur Faces Detected : 2
-----------------------------------

Image: 100_low.jpg
Blur Level : low
Blur Faces Detected   : 2
Deblur Faces Detected : 2
-----------------------------------


... (408 outputs left out)

100%|█████████▉| 621/624 [00:51<00:00, 11.49it/s]


Image: 99_very_high.jpg
Blur Level : very_high
Blur Faces Detected   : 0
Deblur Faces Detected : 1
-----------------------------------

Image: 9_high.jpg
Blur Level : high
Blur Faces Detected   : 1
Deblur Faces Detected : 1
-----------------------------------

Image: 9_low.jpg
Blur Level : low
Blur Faces Detected   : 1
Deblur Faces Detected : 1
-----------------------------------


100%|██████████| 624/624 [00:51<00:00, 12.13it/s]


Image: 9_medium.jpg
Blur Level : medium
Blur Faces Detected   : 1
Deblur Faces Detected : 1
-----------------------------------

Image: 9_very_high.jpg
Blur Level : very_high
Blur Faces Detected   : 0
Deblur Faces Detected : 1
-----------------------------------

===== FACE DETECTION RESULTS =====

BLUR LEVEL : LOW
Blur Faces Detected   : 193
Deblur Faces Detected : 186
Improvement           : -3.63%

BLUR LEVEL : MEDIUM
Blur Faces Detected   : 168
Deblur Faces Detected : 179
Improvement           : 6.55%

BLUR LEVEL : HIGH
Blur Faces Detected   : 122
Deblur Faces Detected : 178
Improvement           : 45.90%

BLUR LEVEL : VERY_HIGH
Blur Faces Detected   : 49
Deblur Faces Detected : 179
Improvement           : 265.31%


In [27]:
!zip -r saved_models.zip saved_models

!zip -r output_images.zip output_images

  adding: saved_models/ (stored 0%)
  adding: saved_models/unet_epoch_22.pth (deflated 7%)
  adding: saved_models/unet_epoch_24.pth (deflated 7%)
  adding: saved_models/unet_epoch_30.pth (deflated 7%)
  adding: saved_models/unet_epoch_20.pth (deflated 7%)
  adding: saved_models/unet_epoch_7.pth (deflated 8%)
  adding: saved_models/unet_epoch_2.pth (deflated 8%)
  adding: saved_models/unet_epoch_23.pth (deflated 7%)
  adding: saved_models/unet_epoch_6.pth (deflated 8%)
  adding: saved_models/unet_epoch_1.pth (deflated 8%)
  adding: saved_models/unet_epoch_13.pth (deflated 8%)
  adding: saved_models/unet_epoch_3.pth (deflated 8%)
  adding: saved_models/unet_epoch_5.pth (deflated 8%)
  adding: saved_models/unet_epoch_21.pth (deflated 7%)
  adding: saved_models/unet_epoch_4.pth (deflated 8%)
  adding: saved_models/unet_epoch_14.pth (deflated 7%)
  adding: saved_models/unet_epoch_8.pth (deflated 8%)
  adding: saved_models/unet_epoch_29.pth (deflated 7%)
  adding: saved_models/unet_epoch_17.